<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/resume_agent_lab2_ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab — Build a Resume-Screening Agent with Pydantic

**Companion to:** *AI Agents & Agentic AI* (slides 13–18)

### What you will build

A small agent that takes **messy CV text** and turns it into a **validated, typed record** you could
put straight into a database — then scores each candidate against a job description.

```
  messy text  ──▶  LLM (schema-constrained)  ──▶  Pydantic validation  ──▶  trusted data
                            ▲                              │
                            └──────  "you got X wrong" ◀────┘   (the repair loop)
```

### How this notebook is documented

**Every code cell opens with a banner** telling you four things:

```
# CELL n · what this cell is for
# INPUT   : what goes in
# OUTPUT  : what comes out
# AI USED : YES / NO  — and why
```

**Every function has a docstring** in the same format. So at any point you can answer the two questions
that matter most when reading agent code: *what does this take and give back?* and *is a language model
involved here, or is this ordinary deterministic Python?*

### Who this is for

- **Non-coders:** every code cell has a plain-English explanation above it. Run the cells in order
  (`Shift + Enter`) and read the output. You do not need to write anything.
- **Coders:** the exercises at the end are where the real learning is.

### How to run it

| Option | What you need |
|---|---|
| **Live model** (recommended) | A free open-source model run by **Ollama inside this Colab** — no API key, no account |
| **Offline mode** | Nothing at all. Set `OFFLINE_MODE = True` in Step 0 and everything still runs |

### Free local model via Ollama (no key needed)

1. In Colab: **Runtime → Change runtime type → T4 GPU** (free). CPU also works, just slower.
2. Run Cell 1 — it installs Ollama into the Colab VM.
3. Run Cell 2 — it starts the Ollama server and downloads the model (`qwen2.5:3b`, ~2 GB, one time per session).

Nothing leaves the Colab VM, and there are no rate limits. The model download takes 1–3 minutes.
If anything goes wrong, flip to offline mode.

---

## Where AI is used — the whole map, up front

This is the most important table in the notebook. Most of what looks like "AI code" is not AI at all.

| # | Component | Kind | AI? | Why |
|---|---|---|---|---|
| 1 | `Experience`, `Candidate` | Pydantic model | ❌ NO | A data contract. Pure Python rules. |
| 2 | `tidy_skills`, `check_email` | Validator | ❌ NO | Deterministic cleaning + checking. |
| 3 | `RESUMES` | Data | ❌ NO | The raw input documents. |
| 4 | `call_model` | **LLM call** | ✅ **YES** | **The only place an LLM is invoked.** |
| 5 | `offline_response` | Stub | ❌ NO | Fake LLM output so the lab runs with no key. |
| 6 | `extract_candidate` | **Agent** | 🟡 PARTLY | Its *loop, validation and retry* are plain Python; it calls `call_model` for the thinking. |
| 7 | `extract_strict` | **Agent** | 🟡 PARTLY | Same shape, stricter contract, to make failures visible. |
| 8 | `skill_overlap` | **Tool** | ❌ NO | Set arithmetic. Never ask a language model to count. |
| 9 | `FitAssessment` | Pydantic model | ❌ NO | Constrains the *judgement* to four allowed values. |
| 10 | `assess_fit` | **Agent** | 🟡 PARTLY | Runs the tool (no AI), then asks the LLM to judge (AI), then validates (no AI). |
| 11 | Batch loop + pandas | Orchestration | ❌ NO | Ordinary data engineering. |

**Read the pattern:** exactly one function talks to a model. Everything around it — the contract, the tool,
the loop, the retry, the error handling — is software you fully control. That ratio is what a real agent
looks like, and it is why the engineering discipline you already have transfers directly.

---

## Step 0 — Setup

Install the libraries. `pydantic` does the validating, `ollama` is the Python client (and we also install the Ollama server itself),
`pandas` gives us a table at the end.

> **Note:** `%pip install` inside a notebook installs into the kernel that is actually running. If you see
> a "restart the kernel" message, do it, then continue from here.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 1 · Install dependencies (Python packages + the Ollama server)
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : nothing
# OUTPUT  : pydantic, ollama (client), pandas, and the Ollama server binary
#           installed into this Colab VM; prints "done"
# AI USED : NO — plain package installation
# ══════════════════════════════════════════════════════════════════════════

%pip install -q "pydantic>=2.7" ollama pandas
!apt-get -qq install -y zstd pciutils > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!ollama --version
print("done")

done


### Choose your mode

- `OFFLINE_MODE = False` → uses the real free model, served by Ollama in this Colab. No key needed.
- `OFFLINE_MODE = True`  → uses a canned, realistic response. **No model download, no server, no waiting.**

Everything else in this notebook is identical in both modes, so nobody gets stuck.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 2 · Configuration — start Ollama, or pick offline mode
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : OFFLINE_MODE flag below. In live mode, Cell 1 must have run.
# OUTPUT  : module-level globals used by the rest of the notebook —
#             OFFLINE_MODE : bool
#             MODEL        : str   the Ollama model tag we will call
#             client       : ollama.Client, or None when offline
# AI USED : NO — this only *sets up* the connection (starts the server and
#           downloads the model weights). Nothing is asked of a model here.
#           The first actual AI call happens in CELL 7.
# ══════════════════════════════════════════════════════════════════════════

OFFLINE_MODE = False        # <-- flip to True to skip Ollama entirely

# Free, open-weight model. Good JSON/structured-output quality for its size.
# Alternatives:  "llama3.2:3b" (similar size) · "qwen2.5:7b" / "llama3.1:8b" (better, needs T4 GPU)
MODEL = "qwen2.5:3b"

import json, os, subprocess, time, urllib.request

client = None
if not OFFLINE_MODE:
    import ollama

    def _server_up() -> bool:
        try:
            urllib.request.urlopen("http://127.0.0.1:11434", timeout=1)
            return True
        except Exception:
            return False

    # Start `ollama serve` in the background if it is not already running
    if not _server_up():
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(30):
            if _server_up():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama server did not start. Did Cell 1 finish? Try OFFLINE_MODE = True.")

    # Download the model weights (cached after the first time in this session)
    print(f"Pulling {MODEL} (first time takes 1-3 min)...")
    subprocess.run(["ollama", "pull", MODEL], check=True)

    client = ollama.Client(host="http://127.0.0.1:11434")
    print("Live mode — model:", MODEL, "(running locally via Ollama)")
else:
    print("Offline mode — using canned responses. No model needed.")

Pulling qwen2.5:3b (first time takes 1-3 min)...
Live mode — model: qwen2.5:3b (running locally via Ollama)


---
## Step 1 — Design the contract *before* you write a prompt

This is the habit worth stealing from today: **decide what shape the answer must have, first.**

A Pydantic model is that shape. Read it like a form:

- `full_name: str` — required, must be text
- `years_experience: float = Field(ge=0, le=60)` — required, a number between 0 and 60
- `email: Optional[str] = None` — *optional*; if the CV does not have one, `None` is a valid answer
- `level: Literal["junior", "mid", "senior", "lead"]` — must be **exactly one of these four strings**

Two design decisions to notice, because they are the ones people get wrong:

1. **`Optional` is not laziness.** Every field you mark *required* is a field the model must invent if the
   document does not contain it — and models would rather invent than fail. If a CV might genuinely lack
   a phone number, make it optional.
2. **`description=` is not a comment.** It is sent to the model inside the JSON Schema. It is prompt
   engineering that lives next to the field it affects.

**No AI in this cell.** We are writing rules, not asking anything.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 3 · The data contract — two Pydantic models + two validators
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : nothing at runtime (this cell only DEFINES classes)
# OUTPUT  : two importable classes —
#             Experience : one job entry
#             Candidate  : the full validated record we want from a CV
#           Each class can then do three jobs:
#             Candidate(**data)                -> validate a dict
#             Candidate.model_validate_json(s) -> validate a JSON string
#             Candidate.model_json_schema()    -> the JSON Schema we send an LLM
# AI USED : NO — pure Python rules. This is the referee, not the player.
# ══════════════════════════════════════════════════════════════════════════

from typing import Literal, Optional
from pydantic import BaseModel, Field, field_validator, ValidationError


class Experience(BaseModel):
    """One job on the CV.

    INPUT  : company (str), title (str), years (float, 0-50)
    OUTPUT : a validated Experience object
    AI USED: NO
    """
    company: str = Field(description="Employer name, as written on the CV")
    title: str = Field(description="Job title held at that employer")
    years: float = Field(ge=0, le=50, description="Duration in years, e.g. 2.5 for 2 years 6 months")


class Candidate(BaseModel):
    """The validated record we want out of a messy CV.

    INPUT  : a dict or JSON string with the fields below
    OUTPUT : a Candidate object with guaranteed types and ranges,
             OR a ValidationError listing every field that broke a rule
    AI USED: NO — but note that every `description=` string below IS shipped
             to the model inside the JSON Schema, so those descriptions are
             prompt engineering that happens to live in the class.
    """

    # ---- identity -------------------------------------------------------
    full_name: str = Field(min_length=2, max_length=80,
                           description="Candidate's full name")
    email: Optional[str] = Field(default=None,          # Optional -> null is a valid answer
                                 description="Email address if present on the CV, else null")
    location: Optional[str] = Field(default=None,
                                    description="City and/or country if stated, else null")

    # ---- experience -----------------------------------------------------
    # le=60 is the guardrail that catches the classic date hallucination
    # (the model reading "2019" as a duration instead of a year)
    years_experience: float = Field(ge=0, le=60,
                                    description="TOTAL professional experience in years, summed "
                                                "across all roles. Estimate from dates if not stated.")

    # Literal is the highest-value constraint with LLMs: it turns "whatever
    # word it felt like" into one of exactly four values you can GROUP BY
    level: Literal["junior", "mid", "senior", "lead"] = Field(
        description="junior: <2y, mid: 2-5y, senior: 5-10y, lead: 10y+")

    # ---- lists ----------------------------------------------------------
    skills: list[str] = Field(default_factory=list,
                              description="Technical skills and tools, lowercase, no duplicates")
    experience: list[Experience] = Field(default_factory=list,
                                         description="Each role held, most recent first")

    # ---- custom rules: clean the data WHILE checking it ------------------

    @field_validator("skills")
    @classmethod
    def tidy_skills(cls, v: list[str]) -> list[str]:
        """Normalise the skills list.

        INPUT  : v — list[str], possibly messy: [" Python ", "SQL", "python"]
        OUTPUT : list[str], trimmed, lowercased, de-duplicated, order preserved
        AI USED: NO — deterministic string handling. Whatever this returns
                 becomes the stored value, so a validator is also a cleaner.
        """
        seen, out = set(), []
        for s in v:
            s = s.strip().lower()
            if s and s not in seen:
                seen.add(s)
                out.append(s)
        return out

    @field_validator("email")
    @classmethod
    def check_email(cls, v):
        """Check and normalise the email address.

        INPUT  : v — str or None
        OUTPUT : lowercased, trimmed str; or None if there was no email
        RAISES : ValueError if the string is not plausibly an email —
                 which Pydantic turns into a ValidationError naming the field
        AI USED: NO
        """
        if v is None:                       # None is legal: the field is Optional
            return None
        v = v.strip().lower()
        if "@" not in v or "." not in v.split("@")[-1]:
            raise ValueError("not a valid email address")
        return v


print("Contract defined. Fields:", list(Candidate.model_fields))

Contract defined. Fields: ['full_name', 'email', 'location', 'years_experience', 'level', 'skills', 'experience']


### Watch the contract do its job

Before we involve any AI at all, let's prove the model is actually enforcing something.

The first cell is clean data. The second deliberately breaks **three** rules at once — and notice that
Pydantic reports **all three**, each with the field name and a plain-English reason. Those messages are the
raw material for the repair loop later.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 4 · Prove the contract works — VALID input
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : a hand-written dict of candidate fields (deliberately messy)
# OUTPUT  : a validated Candidate, printed as JSON. Watch two things happen:
#             - years_experience "7" (a string) is COERCED to 7.0 (a float)
#             - skills are trimmed, lowercased and de-duplicated by tidy_skills
# AI USED : NO — no model involved. This is Pydantic alone.
# ══════════════════════════════════════════════════════════════════════════

good = Candidate(
    full_name="Asha Menon",
    email="ASHA.MENON@example.com ",                 # uppercase + trailing space
    years_experience="7",                            # a string -> coerced to 7.0
    level="senior",
    skills=[" Python ", "SQL", "python", "Spark"],   # dupes + whitespace + case
)

print("VALID:")
print(good.model_dump_json(indent=2))

VALID:
{
  "full_name": "Asha Menon",
  "email": "asha.menon@example.com",
  "location": null,
  "years_experience": 7.0,
  "level": "senior",
  "skills": [
    "python",
    "sql",
    "spark"
  ],
  "experience": []
}


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 5 · Prove the contract works — INVALID input
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : a dict that breaks three rules on purpose
# OUTPUT  : a ValidationError, unpacked into one line per broken rule.
#           err["loc"] = which field, err["msg"] = why, in plain English.
# AI USED : NO
# WHY IT MATTERS : these error messages are exactly what we will feed back
#           to the model in CELL 8 to make it correct itself.
# ══════════════════════════════════════════════════════════════════════════

try:
    Candidate(
        full_name="A",                 # too short          (min_length=2)
        years_experience=2019,         # out of range       (le=60)  <- date hallucination
        level="rockstar",              # not an allowed value (Literal)
    )
except ValidationError as e:
    print(f"Pydantic found {e.error_count()} problems:\n")
    for err in e.errors():
        print(f"  field: {err['loc'][0]:<18} problem: {err['msg']}")

Pydantic found 3 problems:

  field: full_name          problem: String should have at least 2 characters
  field: years_experience   problem: Input should be less than or equal to 60
  field: level              problem: Input should be 'junior', 'mid', 'senior' or 'lead'


> **Point to remember.** `years_experience=2019` is the interesting one. Without the `le=60` constraint
> that value would sail into your database and silently corrupt a report. A crash is your friend; a silent
> wrong value is the enemy.

---
## Step 2 — The messy input

Real CVs are inconsistent: different headings, dates in different formats, experience stated as a range
rather than a number. Here are three, deliberately varied — a mid/senior engineer, a fresher, and a lead.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 6 · The raw input documents
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : nothing
# OUTPUT  : RESUMES — dict[str, str] mapping a short key to raw CV text.
#           Three deliberately different formats:
#             "asha"  : structured, dated, senior engineer
#             "ravi"  : fresher, experience only as internships in months
#             "maria" : prose, obfuscated email ("AT ... DOT"), 15y lead
# AI USED : NO — this is just data. It is the UNSTRUCTURED side of the problem.
# ══════════════════════════════════════════════════════════════════════════

RESUMES = {

"asha": """
ASHA MENON
Bangalore, India  |  asha.menon@example.com  |  +91 98xxx xxxxx

SUMMARY
Data engineer. Built and ran batch + streaming pipelines for retail analytics.

EXPERIENCE
Senior Data Engineer, Zeta Retail Analytics          Mar 2021 - Present
  - Own the Spark/Airflow pipeline moving ~4TB/day
  - Migrated the warehouse from Redshift to Snowflake
Data Engineer, Kite Labs                             Jul 2019 - Feb 2021
  - Built ingestion jobs in Python; dbt models for finance

SKILLS: Python, SQL, Spark, Airflow, dbt, Snowflake, AWS
""",

"ravi": """
Ravi Kulkarni
ravi.k@example.com

Objective: Seeking my first full-time role in software.

Education: B.E. Computer Science, Pune, 2025.

Internships:
  * Backend intern at Nimbus Softech (6 months, Jan-Jun 2025) - Django REST APIs, PostgreSQL
  * Data science intern, college research lab (4 months) - pandas, scikit-learn

Technical: python, java, sql, git, basic docker
""",

"maria": """
MARIA SANTOS - Machine Learning Lead
Lisbon | maria.santos AT example DOT com

15 years in industry. Currently leading a team of 9 ML engineers at Corvo AI (since 2018),
where I own the recommendation platform. Before that: Staff ML Engineer at Praia Systems
(2014-2018) and Software Engineer at the same company (2010-2014).

Stack: Python, PyTorch.
""",
}

print(f"{len(RESUMES)} resumes loaded:", list(RESUMES))
print("\n--- example: ravi ---")
print(RESUMES["ravi"])

3 resumes loaded: ['asha', 'ravi', 'maria']

--- example: ravi ---

Ravi Kulkarni
ravi.k@example.com

Objective: Seeking my first full-time role in software.

Education: B.E. Computer Science, Pune, 2025.

Internships:
  * Backend intern at Nimbus Softech (6 months, Jan-Jun 2025) - Django REST APIs, PostgreSQL
  * Data science intern, college research lab (4 months) - pandas, scikit-learn

Technical: python, java, sql, git, basic docker



---
## Step 3 — Ask the model, but constrain it to your schema

**This is the cell where AI finally enters the notebook.** Everything before it was contracts and data.

The important line is:

```python
"schema": Candidate.model_json_schema()
```

The **same class** you validate with is the class the model is constrained by. There is no second place
to edit, so the prompt and the code can never drift apart.

`response_format={"type": "json_schema", ...}` is not a polite request — on models that support it, the
decoder is restricted to tokens that keep the output valid against your schema. Free-text answers become
structurally impossible.

Three things live in this cell, and only one of them is AI:

| Function | AI? | Role |
|---|---|---|
| `offline_response` | ❌ NO | Fake, fixed LLM output so the lab runs without a key |
| `call_model` | ✅ **YES** | The single point where a language model is invoked |
| `SYSTEM_PROMPT` | ❌ NO | Just a string — but it is the instructions the AI is given |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 7 · The AI boundary — one prompt, one stub, one model call
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : uses the globals set in CELL 2 (client, MODEL, OFFLINE_MODE)
# OUTPUT  : three importable names —
#             SYSTEM_PROMPT    : str      the standing instructions for the LLM
#             offline_response : function fake LLM output (no key needed)
#             call_model       : function THE single AI entry point
# AI USED : YES — call_model() is the ONLY function in this notebook that
#           sends anything to a language model. Everything else is your code.
# ══════════════════════════════════════════════════════════════════════════

# --- The instructions given to the AI. Not code — prompt text. -------------
# Note what it does: restricts the model to facts in the document, tells it to
# prefer null over guessing, and defines how to count internships. Ambiguity
# here shows up as inconsistent extractions later.
SYSTEM_PROMPT = (
    "You extract structured candidate data from resume text. "
    "Use ONLY facts present in the text. If a field is not stated, use null "
    "rather than guessing. For years_experience, sum the durations of professional "
    "roles (internships count as half). Reply with JSON only."
)


# def offline_response(resume_text: str) -> str:
#     """Stand-in for the LLM so the notebook runs with no API key.

#     INPUT  : resume_text — str, the raw CV text (used only to pick which
#              canned answer to return)
#     OUTPUT : str — a JSON string shaped like a real model reply
#     AI USED: NO — this is a hard-coded lookup. It exists so that nobody in
#              the class is blocked by a missing key or a rate limit. It is
#              deliberately NOT clever: same input, same output, every time.
#     """
#     if "ASHA" in resume_text:
#         return json.dumps({
#             "full_name": "Asha Menon", "email": "asha.menon@example.com",
#             "location": "Bangalore, India", "years_experience": 6.5, "level": "senior",
#             "skills": ["Python", "SQL", "Spark", "Airflow", "dbt", "Snowflake", "AWS"],
#             "experience": [
#                 {"company": "Zeta Retail Analytics", "title": "Senior Data Engineer", "years": 4.5},
#                 {"company": "Kite Labs", "title": "Data Engineer", "years": 1.6},
#             ]})
#     if "Ravi" in resume_text:
#         return json.dumps({
#             "full_name": "Ravi Kulkarni", "email": "ravi.k@example.com", "location": None,
#             "years_experience": 0.4, "level": "junior",
#             "skills": ["python", "java", "sql", "git", "docker"],
#             "experience": [
#                 {"company": "Nimbus Softech", "title": "Backend Intern", "years": 0.25},
#                 {"company": "College Research Lab", "title": "Data Science Intern", "years": 0.17},
#             ]})
#     return json.dumps({
#         "full_name": "Maria Santos", "email": None, "location": "Lisbon",
#         "years_experience": 15.0, "level": "lead",
#         "skills": ["Python", "PyTorch", "Kubernetes", "Ray", "MLOps", "Go"],
#         "experience": [
#             {"company": "Corvo AI", "title": "Machine Learning Lead", "years": 7.0},
#             {"company": "Praia Systems", "title": "Staff ML Engineer", "years": 4.0},
#             {"company": "Praia Systems", "title": "Software Engineer", "years": 4.0},
#         ]})


def call_model(messages: list[dict], schema: type[BaseModel]) -> str:
    """### THE AI CALL ### — one request to the LLM, constrained to `schema`.

    INPUT  : messages — list[dict], the chat history so far, e.g.
                 [{"role": "system",  "content": "..."},
                  {"role": "user",    "content": "<resume text>"}]
             schema   — type[BaseModel], the Pydantic class the reply must match.
                        Converted to JSON Schema and sent WITH the request, so
                        the decoder can only emit tokens that keep the JSON valid.

    OUTPUT : str — the model's raw reply as a JSON string.
             IMPORTANT: this string is NOT yet validated. It is untrusted text
             until someone calls schema.model_validate_json() on it.

    AI USED: YES. This is the only AI in the notebook. Every "intelligent"
             behaviour you see later — reading a CV, judging a candidate —
             ultimately happens inside this one function.
             In OFFLINE_MODE it short-circuits to offline_response() instead.
    """
    # if OFFLINE_MODE:
    #     # Pick the FIRST user message (the resume) so retries stay consistent —
    #     # on a retry the last message is an error report, not a CV.
    #     first_user = next(m["content"] for m in messages if m["role"] == "user")
    #     return offline_response(first_user)

    resp = client.chat(
        model=MODEL,
        messages=messages,
        format=schema.model_json_schema(),   # <-- the single source of truth; Ollama constrains decoding to it
        options={"temperature": 0},          # extraction wants determinism, not creativity
    )
    return resp["message"]["content"]


print("AI boundary defined. call_model() is the only function that talks to a model.")
print("\nThe schema we send the model has these top-level fields:")
print(list(Candidate.model_json_schema()["properties"]))

AI boundary defined. call_model() is the only function that talks to a model.

The schema we send the model has these top-level fields:
['full_name', 'email', 'location', 'years_experience', 'level', 'skills', 'experience']


> **Portability note (worth knowing).** `model_json_schema()` puts nested models in a `$defs` section and
> refers to them with `$ref`. Ollama (via llama.cpp grammars), OpenAI and Groq resolve those fine; a few providers
> do not. If you ever see a provider complain about `$ref`, the fix is to inline the definitions before sending — or
> flatten the model so it has no nested classes. Also note that small local models are less reliable than hosted
> ones: the schema forces *valid JSON*, but values can still be wrong — which is exactly why the Pydantic
> validation and repair loop below exists.

### The agent: extract, validate, and repair

This is the pattern from slide 18, and it is where the word *agent* is earned. Read the flow in English:

1. Ask the model for the candidate record. *(AI)*
2. Try to validate it. If it validates, **return it** — everything downstream is now safe. *(not AI)*
3. If it fails, take Pydantic's error list, **paste it back into the conversation**, and ask again. *(not AI —
   but it makes the next AI call much more likely to succeed)*
4. Give up after a couple of tries and raise loudly. Never return half-filled data. *(not AI)*

Step 3 is why this is an agent and not just an API call: it observes the result of its own action and
corrects itself. Crucially the signal is **external** — a deterministic schema check, not the model
grading its own homework.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 8 · THE AGENT — extract a Candidate, with a self-repair loop
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : nothing at runtime (defines one function)
# OUTPUT  : extract_candidate — the agent function, documented below
# AI USED : PARTLY — the loop, the validation and the retry logic are plain
#           Python that you control. Only the call_model() line is AI.
# ══════════════════════════════════════════════════════════════════════════

def extract_candidate(resume_text: str, retries: int = 2, verbose: bool = True) -> Candidate:
    """AGENT — turn one messy CV into one validated Candidate record.

    INPUT  : resume_text — str, the raw unstructured CV text
             retries     — int, how many repair attempts after the first try
                           (default 2, so at most 3 model calls per CV)
             verbose     — bool, print each failed attempt so you can watch
                           the repair loop work

    OUTPUT : Candidate — a fully validated object. Every field is guaranteed
             to exist, have the right type, and be inside its allowed range.
    RAISES : RuntimeError if the model could not produce valid output after
             all retries. Deliberate: we fail loudly rather than return a
             half-filled record. Route these to a human queue.

    AI USED: PARTLY, and it is worth being precise about where —
               AI      : call_model(...)  decides what the fields should be
               NOT AI  : the for-loop, the bound on retries,
                         Candidate.model_validate_json() (the check),
                         building the correction message,
                         the final raise
             The agentic behaviour comes from wrapping an AI call in a loop
             that can see whether the last attempt worked.

    THE LOOP, in four steps:
        1. PERCEIVE : messages carry the goal + everything learned so far
        2. REASON   : the model proposes a record            <- AI
        3. CHECK    : Pydantic validates it                  <- not AI
        4. OBSERVE  : on failure, the error goes back in and we loop
    """
    # 1. PERCEIVE — build the conversation: standing instructions + the document
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": resume_text},
    ]

    for attempt in range(retries + 1):          # bounded: never `while True`

        # 2. REASON — the only AI step. Returns raw, untrusted JSON text.
        raw = call_model(messages, Candidate)

        # 3. CHECK — deterministic. Either we get a real object or an error.
        try:
            candidate = Candidate.model_validate_json(raw)
            if verbose and attempt:
                print(f"   recovered on attempt {attempt + 1}")
            return candidate                    # <-- everything below here is trusted data

        # 4. OBSERVE — feed the precise failure back so the next attempt can fix it
        except ValidationError as e:
            if verbose:
                print(f"   attempt {attempt + 1} failed validation:")
                for err in e.errors():
                    print(f"      - {'.'.join(str(x) for x in err['loc'])}: {err['msg']}")

            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content":
                "Your JSON failed validation with these errors:\n"
                f"{json.dumps(e.errors(), default=str, indent=2)}\n"
                "Return corrected JSON only, obeying the schema exactly."})

    # Out of retries. Fail loudly — a half-filled record is worse than none.
    raise RuntimeError("Could not produce a valid Candidate after retries — send to a human queue.")


print("Agent defined:", extract_candidate.__name__)

Agent defined: extract_candidate


sample resume for below cell:

"asha": """
ASHA MENON
Bangalore, India  |  asha.menon@example.com  |  +91 98xxx xxxxx

SUMMARY
Data engineer. Built and ran batch + streaming pipelines for retail analytics.

EXPERIENCE
Senior Data Engineer, Zeta Retail Analytics          Mar 2021 - Present
  - Own the Spark/Airflow pipeline moving ~4TB/day
  - Migrated the warehouse from Redshift to Snowflake
Data Engineer, Kite Labs                             Jul 2019 - Feb 2021
  - Built ingestion jobs in Python; dbt models for finance

SKILLS: Python, SQL, Spark, Airflow, dbt, Snowflake, AWS

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 9 · Run the agent on one CV
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : RESUMES["asha"] — raw CV text (str)
# OUTPUT  : `asha`, a validated Candidate object, printed as JSON and then
#           accessed as ordinary Python attributes
# AI USED : YES, indirectly — extract_candidate() calls call_model() inside.
#           Expect 1 model call if it validates first time.
# ══════════════════════════════════════════════════════════════════════════

ravi = extract_candidate(RESUMES["ravi"])

print(ravi.model_dump_json(indent=2))
print()
print("Now it is ordinary Python — typed, autocompleted, safe:")
print("  name          :", ravi.full_name)          # str, guaranteed present
print("  level         :", ravi.level)              # one of exactly four values
print("  total years   :", ravi.years_experience)   # float in [0, 60]
print("  top 3 skills  :", ravi.skills[:3])         # list[str], cleaned
print("  most recent   :", ravi.experience[0].title, "@", asha.experience[0].company)

{
  "full_name": "Ravi Kulkarni",
  "email": "ravi.k@example.com",
  "location": null,
  "years_experience": 2.5,
  "level": "junior",
  "skills": [
    "python",
    "java",
    "sql",
    "git",
    "basic docker"
  ],
  "experience": [
    {
      "company": "Nimbus Softech",
      "title": "Backend intern",
      "years": 0.5
    },
    {
      "company": "college research lab",
      "title": "Data science intern",
      "years": 0.5
    }
  ]
}

Now it is ordinary Python — typed, autocompleted, safe:
  name          : Ravi Kulkarni
  level         : junior
  total years   : 2.5
  top 3 skills  : ['python', 'java', 'sql']
  most recent   : Backend intern @ Nimbus Softech


### Break it on purpose

The repair loop is only interesting if you can see it working. Below we deliberately hand the model a
**stricter** contract that the first answer is unlikely to satisfy — `skills` must contain at least 8 items,
and `location` is now required.

Run it and watch the loop: attempt 1 fails with a named error, the error goes back to the model, and a
live model usually recovers on attempt 2.

*(In offline mode the canned response is fixed, so this cell will exhaust its retries and raise — that is
also a correct outcome, and it shows you the "give up loudly" branch. Either way, continue to Step 4.)*

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 10 · Make it fail on purpose — a deliberately harder contract
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : RESUMES["asha"] — the same CV as before
# OUTPUT  : either a StrictCandidate (live mode, usually recovers on try 2)
#           or a printed RuntimeError (offline mode, retries exhausted).
#           Both outcomes are correct and both are worth seeing.
# AI USED : YES, indirectly — up to 3 call_model() calls, one per attempt.
# TEACHING POINT : the ONLY thing that changed is the contract. Same CV,
#           same model, same prompt — a stricter schema changes the behaviour.
# ══════════════════════════════════════════════════════════════════════════

class StrictCandidate(Candidate):
    """A harder version of the contract, used only to force a failure.

    INPUT  : same as Candidate
    OUTPUT : a StrictCandidate — but only if location is present AND there
             are at least 8 skills
    AI USED: NO — inherits everything from Candidate and tightens two fields
    """
    location: str = Field(description="City and country — REQUIRED, infer if necessary")
    skills: list[str] = Field(min_length=8,
                              description="At least 8 skills; include adjacent tools you can infer")


def extract_strict(resume_text: str, retries: int = 2) -> "StrictCandidate":
    """AGENT (variant) — same loop as extract_candidate, stricter contract.

    INPUT  : resume_text — str, raw CV text
             retries     — int, repair attempts after the first try
    OUTPUT : StrictCandidate on success
    RAISES : RuntimeError when the retries are exhausted
    AI USED: PARTLY — call_model() is the AI; the loop and checks are not.
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": resume_text}]

    for attempt in range(retries + 1):
        raw = call_model(messages, StrictCandidate)          # <- AI
        try:
            return StrictCandidate.model_validate_json(raw)  # <- not AI: the check
        except ValidationError as e:
            print(f"attempt {attempt + 1}: {e.error_count()} error(s)")
            for err in e.errors():
                print(f"   - {'.'.join(str(x) for x in err['loc'])}: {err['msg']}")
            messages += [
                {"role": "assistant", "content": raw},
                {"role": "user", "content": "Fix these validation errors and return JSON only:\n"
                                            + json.dumps(e.errors(), default=str)},
            ]
    raise RuntimeError("gave up — route to a human")


try:
    strict = extract_strict(RESUMES["ravi"])
    print("\nRecovered. Skills now:", strict.skills)
except RuntimeError as e:
    print("\nFailed loudly, as designed:", e)

print(strict.model_dump_json(indent=2))


Recovered. Skills now: ['python', 'java', 'sql', 'git', 'basic docker', 'django rest apis', 'postgresql', 'pandas', 'scikit-learn']
{
  "full_name": "Ravi Kulkarni",
  "email": null,
  "location": "ravi.k@example.com",
  "years_experience": 2.5,
  "level": "junior",
  "skills": [
    "python",
    "java",
    "sql",
    "git",
    "basic docker",
    "django rest apis",
    "postgresql",
    "pandas",
    "scikit-learn"
  ],
  "experience": [
    {
      "company": "Nimbus Softech",
      "title": "Backend intern",
      "years": 0.5
    },
    {
      "company": "college research lab",
      "title": "Data science intern",
      "years": 0.5
    }
  ]
}


---
## Step 4 — Give the agent a tool

So far we have one model call. An **agent** does something with the result and decides what happens next.

Here the tool is a boring Python function — `skill_overlap()` — that compares the candidate's skills to a
job description. Two things are worth stating explicitly:

- **The tool is deterministic. There is no AI in it.** Set matching is a job for code. Give the model the
  reasoning; give your code the arithmetic. If you ask a language model to count, it will sometimes be wrong,
  and you will never know which times.
- **The model's judgement still comes back through a Pydantic model** (`FitAssessment`), so the recommendation
  is one of exactly four values you can filter on — not a paragraph of prose.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 11 · The TOOL and the judgement contract
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : nothing at runtime (defines a job spec, a tool, and a model)
# OUTPUT  : JOB_DESCRIPTION : str            what we are hiring for
#           REQUIRED / NICE : set[str]       the skill bars
#           skill_overlap   : function       THE TOOL (deterministic)
#           FitAssessment   : Pydantic model the shape of the AI's judgement
# AI USED : NO in this cell. The tool is pure Python; FitAssessment is a
#           contract. The AI that USES them appears in CELL 12.
# ══════════════════════════════════════════════════════════════════════════

JOB_DESCRIPTION = """
Role: Senior Data Engineer, Analytics Platform
Must have: Python, SQL, Spark, Airflow, cloud data warehouse (Snowflake or BigQuery)
Nice to have: dbt, Kafka, Terraform
Experience: 5+ years building production data pipelines.
"""

REQUIRED = {"python", "sql", "spark", "airflow", "snowflake"}
NICE = {"dbt", "kafka", "terraform"}


def skill_overlap(skills: list[str]) -> dict:
    """TOOL — compare a candidate's skills against the role. No AI here.

    INPUT  : skills — list[str], the candidate's skills
                      (already lowercased by the tidy_skills validator)

    OUTPUT : dict with four keys, all computed by set arithmetic —
               matched_required  : list[str]  required skills the candidate has
               missing_required  : list[str]  required skills they lack
               bonus             : list[str]  nice-to-haves they have
               required_coverage : float 0-1  fraction of REQUIRED matched

    AI USED: NO — and that is the point. This is the kind of work you must
             NOT delegate to a language model. It is countable, so count it.
             A model asked "how many of these 5 skills match?" will usually
             be right and occasionally be wrong, with no way to tell which.

    USED BY: assess_fit() in CELL 12 runs this FIRST, then hands the result
             to the model as evidence, telling it not to recompute.
    """
    have = {s.lower() for s in skills}
    matched = sorted(REQUIRED & have)
    missing = sorted(REQUIRED - have)
    return {
        "matched_required": matched,
        "missing_required": missing,
        "bonus": sorted(NICE & have),
        "required_coverage": round(len(matched) / len(REQUIRED), 2),
    }


class FitAssessment(BaseModel):
    """The AI's judgement — constrained so it is usable as data, not prose.

    INPUT  : a dict or JSON string produced by the model
    OUTPUT : a validated FitAssessment
    AI USED: NO — this is the contract. The model FILLS it in CELL 12.

    Why each constraint exists:
      recommendation : Literal -> four values you can filter and GROUP BY,
                       instead of "looks quite promising overall"
      score          : ge/le   -> a sortable integer, never 150 or -3
      reasons        : min/max -> forces 1-3 specific points, not an essay
    """
    recommendation: Literal["strong_yes", "yes", "maybe", "no"] = Field(
        description="Hiring recommendation for THIS role")
    score: int = Field(ge=0, le=100, description="Overall fit, 0-100")
    reasons: list[str] = Field(min_length=1, max_length=3,
                               description="1-3 short, specific reasons citing the CV")
    gaps: list[str] = Field(default_factory=list,
                            description="Required skills or experience the candidate lacks")


# Run the tool on its own so you can see it is just a dict of sets
print("TOOL OUTPUT for Ravi:")
print(json.dumps(skill_overlap(ravi.skills), indent=2))

TOOL OUTPUT for Ravi:
{
  "matched_required": [
    "python",
    "sql"
  ],
  "missing_required": [
    "airflow",
    "snowflake",
    "spark"
  ],
  "bonus": [],
  "required_coverage": [
    0.4,
    2
  ]
}


Now the agent step. Watch the four words from slide 1 appear in order — and notice which one is AI:

| Step | What happens | AI? |
|---|---|---|
| **Perceive** | Gather the validated candidate + the job description | ❌ |
| **Act** | Run `skill_overlap()` — your code, not the model's | ❌ |
| **Reason** | Ask the model to judge, *given the tool result as evidence* | ✅ |
| **Check** | Validate the judgement against `FitAssessment` | ❌ |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 12 · THE AGENT WITH A TOOL — assess candidate fit
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : defines assess_fit(), then runs it on `asha`
# OUTPUT  : `fit`, a validated FitAssessment, printed as JSON
# AI USED : PARTLY — 1 model call per attempt. The tool call, the prompt
#           assembly and the validation around it are all plain Python.
# ══════════════════════════════════════════════════════════════════════════

def assess_fit(candidate: Candidate, job: str, retries: int = 2) -> FitAssessment:
    """AGENT — score one validated candidate against one job description.

    INPUT  : candidate — Candidate, ALREADY validated (came out of CELL 8)
             job       — str, the job description text
             retries   — int, repair attempts after the first try

    OUTPUT : FitAssessment — recommendation (one of four), score 0-100,
             1-3 reasons, and a list of gaps. Safe to put in a dataframe.
    RAISES : RuntimeError if no valid assessment after all retries.

    AI USED: PARTLY. In order:
               NOT AI : skill_overlap(candidate.skills)   the tool runs first
               NOT AI : building the prompt from candidate + tool result
               AI     : call_model(...)                   the judgement
               NOT AI : FitAssessment.model_validate_json(...)  the check
               NOT AI : the retry loop

    DESIGN NOTE: the tool runs BEFORE the model, and its output is handed to
    the model as evidence with an explicit instruction not to recompute it.
    That split — code does the counting, the model does the judging — is the
    single most useful habit in this notebook.
    """
    # ---- ACT: run the deterministic tool first ---------------------------
    overlap = skill_overlap(candidate.skills)          # <- not AI

    # ---- PERCEIVE: assemble everything the model needs to judge ----------
    prompt = (
        f"JOB DESCRIPTION:\n{job}\n\n"
        f"CANDIDATE (validated):\n{candidate.model_dump_json(indent=2)}\n\n"
        f"TOOL RESULT — skill_overlap:\n{json.dumps(overlap, indent=2)}\n\n"
        "Assess the fit. Use the tool result for skill coverage; do not recompute it. "
        "Be strict about the 5+ years requirement."
    )
    messages = [
        {"role": "system", "content": "You are a technical recruiter. Reply with JSON only."},
        {"role": "user", "content": prompt},
    ]

    for _ in range(retries + 1):

        # ---- REASON: the AI step ----------------------------------------
        # if OFFLINE_MODE:
        #     # Offline stand-in: derive a plausible judgement from the TOOL output
        #     # so offline users still see the right SHAPE of result. Not AI.
        #     cov, yrs = overlap["required_coverage"], candidate.years_experience
        #     rec = ("strong_yes" if cov >= 0.8 and yrs >= 5 else
        #            "yes" if cov >= 0.6 and yrs >= 5 else
        #            "maybe" if cov >= 0.4 else "no")
        #     raw = json.dumps({
        #         "recommendation": rec, "score": int(cov * 70 + min(yrs, 10) * 3),
        #         "reasons": [f"{int(cov*100)}% of required skills matched",
        #                     f"{yrs} years of experience"],
        #         "gaps": overlap["missing_required"]})
        # else:
        raw = call_model(messages, FitAssessment)          # <- AI

        # ---- CHECK: deterministic validation ----------------------------
        try:
            return FitAssessment.model_validate_json(raw)
        except ValidationError as e:
            messages += [{"role": "assistant", "content": raw},
                         {"role": "user", "content": "Fix: " + json.dumps(e.errors(), default=str)}]

    raise RuntimeError("no valid assessment after retries")


fit = assess_fit(ravi, JOB_DESCRIPTION)
print(fit.model_dump_json(indent=2))

{
  "recommendation": "strong_yes",
  "score": 0,
  "reasons": [
    "The candidate has 2.5 years of experience, which is less than the required 5+ years for the role.",
    "The candidate's skills do not cover all the required skills: Python and SQL are matched, but Airflow, Spark, and Snowflake are missing."
  ],
  "gaps": []
}


---
## Step 5 — Run the batch, get a table

This is the payoff. Three unstructured documents in; one dataframe out, with typed columns you can sort,
filter and hand to a recruiter.

Notice the `try/except` around each candidate: in a real batch, **one bad document must not kill the run**.
Failures get collected and routed to a human, and the good ones still go through.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 13 · BATCH — run both agents over every CV and build a table
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : RESUMES — dict[str, str], three raw CVs
#           JOB_DESCRIPTION — str
# OUTPUT  : df       : pandas.DataFrame, one row per successful candidate,
#                      sorted by score descending
#           failures : list[dict], anything that could not be processed
# AI USED : YES, indirectly — 2 agent calls per CV (extract + assess),
#           so roughly 6 model calls for 3 CVs if nothing needs repairing.
#           The loop, the error handling and the dataframe are plain Python.
# ══════════════════════════════════════════════════════════════════════════

import pandas as pd

rows, failures = [], []

for key, text in RESUMES.items():
    print(f"processing {key} ...")
    try:
        # AGENT 1: unstructured text -> validated Candidate
        cand = extract_candidate(text, verbose=False)
        # AGENT 2: validated Candidate + job -> validated FitAssessment
        fit = assess_fit(cand, JOB_DESCRIPTION)

        rows.append({
            "name": cand.full_name,
            "level": cand.level,
            "years": cand.years_experience,
            "location": cand.location or "-",
            "recommendation": fit.recommendation,
            "score": fit.score,
            "missing": ", ".join(fit.gaps) or "-",
            "skills": ", ".join(cand.skills[:5]),
        })

    except Exception as e:                       # noqa: BLE001
        # One bad document must never kill the batch. Collect and continue.
        failures.append({"key": key, "error": str(e)[:120]})

df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
print()
display(df)

if failures:
    print("\nSent to human review:", failures)

processing asha ...
processing ravi ...
processing maria ...



,name,level,years,location,recommendation,score,missing,skills
0,ASHA MENON,senior,4.0,-,strong_yes,0,-,
1,Ravi Kulkarni,junior,2.5,-,strong_yes,0,-,"python, java, sql, git, basic docker"
2,MARIA SANTOS,lead,15.0,-,strong_yes,0,-,"python, pytorch"


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 14 · Use the result like any other data
# ──────────────────────────────────────────────────────────────────────────
# INPUT   : df — the dataframe built in CELL 13
# OUTPUT  : `shortlist` dataframe printed, and screening_results.csv on disk
# AI USED : NO — the AI's work finished at CELL 13. From here it is ordinary
#           data engineering, which is exactly the point: once output is
#           validated, everything downstream is normal software.
# ══════════════════════════════════════════════════════════════════════════

shortlist = df[(df["score"] >= 60) & (df["recommendation"].isin(["yes", "strong_yes"]))]
print("Shortlist:")
display(shortlist[["name", "level", "years", "score", "missing"]])

df.to_csv("screening_results.csv", index=False)
print("\nSaved to screening_results.csv")

Shortlist:


,name,level,years,score,missing



Saved to screening_results.csv


---
## What just happened

Look back at the four words from slide 1:

| Step | Where it was in this notebook | AI? |
|---|---|---|
| **Perceive** | The raw CV text plus the job description went into the prompt | ❌ |
| **Reason** | `call_model()` decided the fields and the recommendation | ✅ |
| **Act** | `skill_overlap()` — your own deterministic Python, run by your code | ❌ |
| **Check** | `model_validate_json()` — and the repair loop when it failed | ❌ |

**Three of the four steps have no AI in them.** That is not a limitation of this lab — it is what a
production agent looks like. The model supplies judgement; your code supplies everything else, and your
code is where reliability, security and auditability live.

### Count of AI calls in a clean run

| Stage | Model calls |
|---|---|
| CELL 9 — extract one CV | 1 |
| CELL 10 — the deliberate failure | up to 3 |
| CELL 12 — assess one candidate | 1 |
| CELL 13 — batch of 3 CVs | ~6 |
| **Everything else** | **0** |

### Points to remember

1. **Design the schema before the prompt.** The shape of the answer is a product decision, not a
   side-effect of prompting.
2. **One class, four jobs** — validation, JSON Schema for the model, editor autocomplete, and documentation.
   Nothing can drift out of sync because there is only one definition.
3. **`Literal` / `Enum` is the highest-value constraint with LLMs.** It turns "whatever word it felt like"
   into a value you can `GROUP BY`.
4. **Optional means optional.** Every required field is a field the model must invent if the document
   lacks it — and it would rather invent than fail.
5. **Feed validation errors back to the model.** Self-correction works when the signal is external
   and precise.
6. **Fail loudly.** Never return a half-filled object. A crash is cheaper than silent corruption.
7. **Keep arithmetic in code.** `skill_overlap()` is deterministic on purpose. Let the model judge; let
   your code count.
8. **Bound everything** — retries, steps, cost. And log the raw output on every failure; those failures
   are your best dataset for fixing the schema.

---

## Exercises

**Warm-up (everyone)**

1. Add a `notice_period_days: Optional[int]` field with `ge=0, le=180`. Re-run CELL 9. Does the model
   correctly return `null` for CVs that do not mention it — or does it invent a number? That behaviour
   is exactly what `Optional` is protecting you from.
2. Change `level` to include `"principal"`. Re-run and see how the extraction shifts for Maria.

**If you write code**

3. Set `temperature=1.0` in `call_model` and run the batch three times. How much do `years_experience`
   and `score` move? This is the non-determinism from slide 7, measured on your own data.
4. Add a `@model_validator(mode="after")` to `Candidate` that raises if `years_experience` is more than
   1.5× the sum of `experience[].years` — a cross-field consistency check the model cannot fake.
5. Add a second tool, `salary_band(level, location)`, and let the model choose between `skill_overlap`
   and `salary_band` using real tool-calling (`tools=[...]`) instead of the hard-wired call in
   `assess_fit`. That is the loop from slide 11.
6. Instrument it: log every prompt, every raw response and every validation error to a JSONL file.
   Then compute your retry rate. Above ~10% means your schema or your descriptions need work — not the model.

**If you do not write code**

7. Rewrite the `description=` text on `years_experience` — for example, change it to *"years at the most
   recent employer only"* — and re-run. Same code, same model, different answer. That is prompt
   engineering, and it lives in the schema.
8. Go through the notebook and, for each cell banner, cover the `AI USED` line and try to guess it before
   reading. If you can do that reliably, you can read almost any agent codebase.
9. Write a Pydantic model (just the field names, types and rules — no other code) for a document you deal
   with at work: an invoice, a claim form, a support ticket. Which fields are genuinely required? That
   exercise is most of the design work in a real extraction project.